# Structural Filtering of Cryptic Clues

**Primary author:** Victoria Winters

**Builds on:**
- *clue_misdirection/notebooks/01_data_cleaning.ipynb* (Victoria — surface-extraction regex, `pattern_from_A()` answer-format validation, double-definition splitting/expansion logic)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Filters `data/clues_raw.csv` to clues satisfying CCC structural constraints — those with non-null definition and answer, an answer whose length/format matches the parenthetical code in the clue, and at least one definition appearing as an intact whole word at the start or end of the surface. Double-definition clues are expanded so each row carries one valid definition. Writes `data/clues_filtered.csv`, the canonical shared upstream artifact for all project components. No WordNet filtering, train/test split assignment, or puzzle-metadata join happens here — those are component-level concerns.

---

## §0 — Imports and Paths

All paths are relative to this notebook's directory via `pathlib`. The shared `clue_utils` module sits next to this notebook and provides the definition-matching logic used in Filter 4.

In [ ]:
# === Imports and Paths ===
import re
import sys
import time
from pathlib import Path

import pandas as pd

# Make `clue_utils.py` (sitting next to this notebook) importable.
sys.path.insert(0, str(Path.cwd()))
from clue_utils import find_definition_in_surface

DATA_DIR = Path("..") / "data"
INPUT_PATH = DATA_DIR / "clues_raw.csv"
OUTPUT_PATH = DATA_DIR / "clues_filtered.csv"

# Track row counts at each stage so the summary cell can report fractions.
filter_log = {}

---

## §1 — Load Raw Clues

Load only the four columns this notebook needs. `keep_default_na=False` with `na_values=[""]` is required because the word `nan` (a valid crossword entry meaning "grandmother") would otherwise be silently coerced to `NaN`.

In [ ]:
# === Load Raw Clues ===
t0 = time.time()
df = pd.read_csv(
    INPUT_PATH,
    usecols=["clue_id", "clue", "answer", "definition"],
    keep_default_na=False,
    na_values=[""],
)
load_runtime = time.time() - t0

filter_log["00_loaded"] = len(df)
print(f"Loaded {len(df):,} rows from {INPUT_PATH} in {load_runtime:.1f}s")

---

## §2 — Filter 1: Drop Null Clue, Definition, or Answer

Three sources (`cru_cryptics`, `nytimes`, `leoedit`) carry no definition field at all and are eliminated wholesale here, alongside individual rows from other sources that lack one or the other field.

In [ ]:
# === Filter 1: Drop Null Clue, Definition, or Answer ===
n_before = len(df)
df = df.dropna(subset=["clue", "definition", "answer"]).copy()
filter_log["01_dropna"] = len(df)
print(f"Dropped {n_before - len(df):,} rows; {len(df):,} remaining")

---

## §3 — Filter 2: Clean the Clue Surface

Three character classes need cleaning out of the surface text before downstream filters can use it. We strip the trailing answer-format code first (e.g. `(4-2-3-3)`) to produce the `surface` column, then apply three rules in order:

- **`/`** — stripped unconditionally; bloggers inserted slashes to mark the boundary between definitions in double-definition clues, but they are not part of the original published surface.
- **`*`** — when `*` is the very first character of the clue and is immediately followed by a capital letter (a marker for specialty themed puzzles), the leading `*` is stripped from the surface so the surface reads as a standalone clue. All other uses of `*` — typically censorship like `b*** hell` or `M*A*S*H` — are left intact in the surface. No rows are dropped on the basis of `*`.
- **`[ ]`** — rows where brackets surround an all-caps sequence (e.g. `[VERVE]`, `[ELGIAN]`) come from extra-word puzzle variants that violate standard cryptic crossword rules, and are excluded. Other bracketed annotations (`[in]`, `[of]`, `['s]`) are kept and the brackets are stripped from the surface.

Finally, any runs of two or more spaces left behind by character removal (e.g. `"word / word"` → `"word  word"`) are collapsed to a single space and leading/trailing whitespace is trimmed. This keeps the surface clean both for Filter 4's edge-match check and for the final `clues_filtered.csv` output.

The original `clue` column is left untouched — it remains the source of truth for the diagnostic cells later in the notebook. Each rule's rationale is documented in `DECISIONS.md`.

In [ ]:
# === Filter 2: Clean the Clue Surface ===

# (a) Strip the trailing answer-format code to produce the surface.
df["surface"] = df["clue"].apply(
    lambda x: re.sub(r"\s*\(\d+(?:[,\s-]+\d+)*\)$", "", x)
)

# (b) Strip "/" unconditionally.
df["surface"] = df["surface"].str.replace("/", "", regex=False)

# (c) Strip the leading "*" from surface when the clue starts with "*" + capital
#     letter (a specialty puzzle marker). All other "*" — typically censorship like
#     "b*** hell" or "M*A*S*H" — is left intact in the surface, and no rows are dropped.
star_themed = df["clue"].str.match(r"\*[A-Z]")
n_star_themed = int(star_themed.sum())
df.loc[star_themed, "surface"] = (
    df.loc[star_themed, "surface"].str.replace(r"^\*", "", regex=True)
)
print(f"Stripped leading '*' from {n_star_themed:,} specialty-themed surfaces; "
      f"no rows dropped")

# (d) Exclude rows containing bracketed all-caps, then strip "[" and "]" from the rest.
bracket_caps = df["clue"].str.contains(r"\[[A-Z]+\]", regex=True, na=False)
n_before = len(df)
df = df[~bracket_caps].copy()
filter_log["03_bracket_excluded"] = len(df)
print(f"Dropped {n_before - len(df):,} rows with bracketed all-caps; "
      f"{len(df):,} remaining")
df["surface"] = df["surface"].str.replace(r"[\[\]]", "", regex=True)

# (e) Collapse any runs of 2+ spaces down to a single space and trim leading/trailing
#     whitespace. Removing "/", "[", "]", and the leading "*" can leave behind double
#     spaces (e.g. "word / word" -> "word  word"); downstream filters and the final
#     output both want clean surface text.
df["surface"] = df["surface"].str.replace(r" {2,}", " ", regex=True).str.strip()

---

## §4 — Filter 3: Validate Answer Length/Format

Cryptic clues end with a parenthetical code such as `(5)`, `(3,4)`, or `(4-2-3-3)` describing the answer's word/hyphen structure. We:

1. Extract the stated format from inside those parentheses.
2. Compute the actual format from the answer string itself with `pattern_from_A()`.
3. Drop rows where either the stated or the computed format is missing, and rows where the two disagree — these are typographic noise in the source data.

`pattern_from_A()` is ported verbatim from `01_data_cleaning.ipynb` since the format-validation semantics must stay identical to the original cleaning pipeline.

In [ ]:
# === Filter 3: Validate Answer Length/Format ===

# (a) Extract the stated format from inside the trailing parens.
stated_format = (
    df["clue"]
    .str.extract(r"\(\s*([\d,\s-]+)\s*\)\s*$", expand=False)
    .str.replace(" ", "", regex=False)
)


def pattern_from_A(s):
    """Compute the format pattern of an answer string, mirroring how the
    clue encodes it in trailing parentheses.

    Spaces become commas (separating word lengths), hyphens are preserved
    within words, and each alphabetic run contributes its length.

    Examples:
        pattern_from_A("PLANT")           -> "5"
        pattern_from_A("TOP-UP")          -> "3-2"
        pattern_from_A("OLD PIANO")       -> "3,5"
        pattern_from_A("JACK-IN-THE-BOX") -> "4-2-3-3"
    """
    if not isinstance(s, str):
        return None
    s = s.upper().strip()
    word_patterns = []
    for word in s.split():
        parts = re.findall(r"[A-Z]+", word)
        if not parts:
            continue
        if "-" in word:
            # Preserve hyphen structure: "TOP-UP" -> "3-2"
            word_patterns.append("-".join(str(len(p)) for p in parts))
        else:
            word_patterns.append(str(len(parts[0])))
    return ",".join(word_patterns) or None


# (b) Compute the actual format from the answer string.
computed_format = df["answer"].apply(pattern_from_A)

# (c) Drop rows with no extractable stated or computed format.
extractable = stated_format.notna() & computed_format.notna()
n_no_format = int((~extractable).sum())
df = df[extractable].copy()
stated_format = stated_format[extractable]
computed_format = computed_format[extractable]

# (d) Drop rows where the two formats disagree.
format_match = stated_format == computed_format
n_mismatch = int((~format_match).sum())
df = df[format_match].copy()

filter_log["04_answer_format_valid"] = len(df)
print(f"Dropped {n_no_format:,} rows with no extractable format")
print(f"Dropped {n_mismatch:,} rows where stated and actual formats disagreed")
print(f"{len(df):,} remaining")

---

## §5 — Filter 4: Parse Double Definitions and Require an Edge Match

Some clues carry multiple alternative definitions in the `definition` field, separated by `/`. We split on `/`, validate each candidate against the surface using the shared `find_definition_in_surface` utility (which handles case, accents, and `<word>'s` apostrophe-s), and keep the row only if at least one valid candidate appears at the *start* or *end* of the surface. Each surviving row is then exploded so it carries exactly one definition.

Using `find_definition_in_surface` here — rather than ad-hoc regex — keeps this filtering step in lockstep with downstream phrase-construction notebooks that use the same utility to place `<t></t>` delimiters.

In [ ]:
# === Filter 4: Parse Double Definitions, Require Edge Match, Expand ===
t0 = time.time()

_slash_splitter = re.compile(r"/+")
_ws_normalizer = re.compile(r"\s+")


def valid_edge_definitions(definition, surface):
    """Return all `/`-separated candidates from `definition` that
    `find_definition_in_surface` locates in `surface`, but only if at
    least one of them sits at the start or end of the surface. If no
    candidate is at an edge, return an empty list (the clue is dropped).
    """
    parts = _slash_splitter.split(str(definition))

    # Clean and dedupe candidates (case-insensitive on cleaned text).
    seen = set()
    cleaned = []
    for p in parts:
        p_clean = _ws_normalizer.sub(" ", p).strip()
        if not p_clean:
            continue
        key = p_clean.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(p_clean)

    located = []
    for cand in cleaned:
        span = find_definition_in_surface(cand, surface)
        if span is not None:
            located.append((cand, span))

    if not located:
        return []

    # Edge check is performed on the whitespace-stripped surface so that
    # leading/trailing spaces in the raw clue text don't disqualify an
    # otherwise edge-anchored definition.
    surf_stripped = surface.strip()
    lstrip_offset = len(surface) - len(surface.lstrip())
    rstrip_end = lstrip_offset + len(surf_stripped)

    has_edge = any(
        start == lstrip_offset or end == rstrip_end
        for _, (start, end) in located
    )
    if not has_edge:
        return []

    return [cand for cand, _ in located]


def_lists = [
    valid_edge_definitions(d, s)
    for d, s in zip(df["definition"], df["surface"])
]
df = df.drop(columns=["definition"]).assign(definition=def_lists)

n_before = len(df)
df = df[df["definition"].map(len) > 0].copy()
filter_log["05_edge_definition"] = len(df)
print(f"Dropped {n_before - len(df):,} rows with no valid edge definition")

df = df.explode("definition", ignore_index=True)
filter_log["06_expanded"] = len(df)

filter4_runtime = time.time() - t0
print(f"Expanded to {len(df):,} (definition, clue) rows in {filter4_runtime:.1f}s")

---

## §6 — Diagnostic: Surviving Rows Containing `[`

Surviving rows whose original `clue` field contains `[`. After Filter 2, these are all rows where the brackets did *not* enclose an all-caps sequence — typically blogger annotations like `[in]`, `[of]`, `['s]`. The brackets have been stripped from the `surface` column for these rows; the `clue` column is shown unmodified for reference. The exclusion rule and full rationale live in `DECISIONS.md`. **No filtering happens at this step.**

In [ ]:
# === Bracket Diagnostic (No Filtering) ===
bracket_mask = df["clue"].str.contains(r"\[", regex=True, na=False)
n_bracket = int(bracket_mask.sum())
print(f"{n_bracket:,} surviving rows contain '[' in the clue field")

with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(df[bracket_mask][["clue_id", "clue", "surface", "definition", "answer"]].head(10))

---

## §7 — Diagnostic: Surviving Rows Containing `*`

Surviving rows whose original `clue` field contains `*`. These fall into two groups: specialty-themed clues where `*` was the very first character followed by a capital letter (the leading `*` has been stripped from the corresponding `surface`), and clues with `*` elsewhere — typically censorship like `b*** hell` or `M*A*S*H`, or blogger annotations like `(Real ner[d])*` — where `*` is left intact in the surface. The `clue` column is shown unmodified for reference. The decision and full rationale live in `DECISIONS.md`. **No filtering happens at this step.**

In [ ]:
# === Asterisk Diagnostic (No Filtering) ===
star_mask = df["clue"].str.contains(r"\*", na=False)
print(f"Rows with '*' in clue: {star_mask.sum():,}")

with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(df[star_mask][["clue_id", "clue", "surface", "definition", "answer"]])

---

## §8 — Diagnostic: Surviving Rows Containing `/`

Surviving rows whose original `clue` field contains `/`. These are double-definition clues where bloggers inserted a slash to mark the boundary between the two definitions. The slash has been stripped from the `surface` column unconditionally; the `clue` column is shown unmodified for reference. The decision and the known collateral case (`clue_id` 9773) live in `DECISIONS.md`. **No filtering happens at this step.**

In [ ]:
# === Slash Diagnostic (No Filtering) ===
slash_mask = df["clue"].str.contains(r"/", na=False)
print(f"Rows with '/' in clue: {slash_mask.sum():,}")

with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
    display(df[slash_mask][["clue_id", "clue", "surface", "definition", "answer"]].head(10))

---

## §9 — Write `clues_filtered.csv`

Per `WORKFLOW.md`, the shared filtered file carries exactly four columns in this order: `clue_id`, `surface`, `definition`, `answer`. The original `clue` text and any puzzle metadata are deliberately omitted — components join those in on demand using `clue_id` (which is no longer unique after multi-definition expansion).

In [ ]:
# === Write clues_filtered.csv ===
out = df[["clue_id", "surface", "definition", "answer"]]
out.to_csv(OUTPUT_PATH, index=False)
output_size_mb = OUTPUT_PATH.stat().st_size / (1024 * 1024)
print(f"Wrote {len(out):,} rows to {OUTPUT_PATH} ({output_size_mb:.1f} MB)")

---

## §10 — Stage-by-Stage Row Counts

A compact table reporting how many rows survived each stage as both an absolute count and a fraction of the previous stage. The keys mirror the `filter_log` entries set throughout the notebook.

In [ ]:
# === Stage-by-Stage Row Counts ===
stages = [
    ("00_loaded",              "Loaded raw rows"),
    ("01_dropna",              "Drop null clue/definition/answer"),
    ("03_bracket_excluded",    "Exclude bracketed all-caps"),
    ("04_answer_format_valid", "Validate answer format"),
    ("05_edge_definition",     "Require edge definition (pre-explode)"),
    ("06_expanded",            "Expand multi-definition rows"),
]
prev = None
for key, label in stages:
    n = filter_log[key]
    if prev is None:
        print(f"{label:42s} {n:>10,}")
    else:
        frac = (n / prev * 100) if prev else 0.0
        print(f"{label:42s} {n:>10,}   ({frac:5.1f}% of previous)")
    prev = n

---

## §11 — Summary

This notebook applies the structural filters required by the CCC pipeline to `data/clues_raw.csv` and writes the canonical shared upstream artifact `data/clues_filtered.csv`.

**What was done**

1. Loaded `clue_id`, `clue`, `answer`, `definition` from `clues_raw.csv` with `keep_default_na=False, na_values=[""]` so the crossword entry "nan" survives.
2. Dropped rows with null `definition` or `answer` (eliminates `cru_cryptics`, `nytimes`, `leoedit` wholesale, plus stray nulls from other sources).
3. Cleaned the clue surface: stripped the trailing answer-format code to produce `surface`, removed `/` unconditionally, stripped the leading `*` for specialty-themed clues (`*` at start followed by a capital letter) while leaving all other `*` intact, and excluded bracketed all-caps rows then stripped remaining `[ ]`. Only the bracketed-all-caps rule drops any rows at this stage.
4. Extracted the stated format from the parens, computed the actual format from the answer with `pattern_from_A()`, and dropped rows with no extractable format and rows where the two formats disagreed.
5. Split the `definition` field on `/`, validated each candidate against the surface with the shared `find_definition_in_surface` utility (whole-word match, case- and accent-insensitive, accepts `<word>'s`), required at least one valid candidate at the start or end of the (whitespace-stripped) surface, then exploded to one row per valid definition.
6. Reported (without filtering) the count of surviving rows whose `clue` contains `[`, `*`, and `/`; the keep/drop decisions live in `DECISIONS.md`.

**Output**

- `data/clues_filtered.csv` — columns `clue_id`, `surface`, `definition`, `answer` (in this order). Row counts and file size are printed by the cells above. `clue_id` is **not unique** in this file: a clue with two valid definitions appears as two rows sharing the same `clue_id`.

**Notes and edge cases**

- Definition matching delegates to `clue_utils.find_definition_in_surface`, the same function used downstream by `02_phrase_construction.ipynb` for `<t></t>` delimiter placement — changes there will affect both filtering and phrase construction and must be documented in `DECISIONS.md`.
- The edge check uses the whitespace-stripped surface, so trailing/leading whitespace in the raw clue text does not disqualify an edge-anchored definition.
- No WordNet filter, train/validate/test split, or puzzle-metadata join is applied here — those are component-level concerns and live in `custom_embedding_model/notebooks/01_wn_filtering_and_split.ipynb` and the puzzle-metadata notebook respectively.

**Runtimes** are printed inline next to the load step and Filter 4 (the only computationally significant stages); the other stages run in well under a second on the full 660K-row input.